# Is the Embedding Contribution Incremental and Robust?

This notebook evaluates the improvement from adding embeddings to characteristics using paired quarterly losses, autocorrelation-robust inference, moving-block bootstrap intervals, and a final-period shuffled-embedding placebo.

## 1. Setup

All inputs are local. The analysis reuses the frozen penalties and permanent test-firm predictions from Notebook 04.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'economic_shock_sensitivity').exists() and (candidate/'full_replication_80q').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')

ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'; SOURCE=MODULE/'results'/'ex_ante_embedding_models'; RESULTS=MODULE/'results'/'incremental_robustness'
RESULTS.mkdir(parents=True,exist_ok=True); sys.path.insert(0,str(MODULE/'src'))
import embedding_rate_models as erm
predictions=pd.read_parquet(SOURCE/'permanent_test_firm_predictions.parquet')
selected=pd.read_parquet(SOURCE/'selected_penalties.parquet').set_index('model')
targets=pd.read_parquet(MODULE/'data'/'processed'/'rolling_rate_targets'/'all_rolling_rate_targets.parquet')
targets=targets[(targets.specification.eq('stmt_ar00'))&targets.eligible]
print(f'Loaded {len(predictions):,} permanent-test predictions.')


## 2. Paired quarterly improvements

For every common test firm, squared error from the combined model is paired with squared error from characteristics alone. Positive normalized improvement means that embeddings reduce error. Pairing prevents coverage differences from creating an apparent gain.

In [ ]:
comparisons=[('binary','economic_q_minus_1','rs_binary_10_plus_chars__economic_q_minus_1','characteristics_6__economic_q_minus_1'),('binary','public_q_minus_2','rs_binary_10_plus_chars__public_q_minus_2','characteristics_6__public_q_minus_2'),('l_min','economic_q_minus_1','rs_l_min_10_plus_chars__economic_q_minus_1','characteristics_6__economic_q_minus_1'),('l_min','public_q_minus_2','rs_l_min_10_plus_chars__public_q_minus_2','characteristics_6__public_q_minus_2')]
paired_rows=[]
for representation,timing,combined_name,char_name in comparisons:
    combined=predictions[predictions.model.eq(combined_name)][['anchor_quarter','permco','target','prediction']].rename(columns={'prediction':'combined_prediction'})
    chars=predictions[predictions.model.eq(char_name)][['anchor_quarter','permco','prediction']].rename(columns={'prediction':'characteristic_prediction'})
    joined=combined.merge(chars,on=['anchor_quarter','permco'],how='inner',validate='one_to_one')
    joined['combined_error2']=(joined.target-joined.combined_prediction)**2; joined['characteristic_error2']=(joined.target-joined.characteristic_prediction)**2
    for quarter,group in joined.groupby('anchor_quarter'):
        denominator=((group.target-group.target.mean())**2).sum()
        paired_rows.append({'representation':representation,'timing':timing,'quarter':quarter,'firms':len(group),'normalized_error_reduction':(group.characteristic_error2.sum()-group.combined_error2.sum())/denominator,'mean_squared_error_reduction':(group.characteristic_error2-group.combined_error2).mean()})
paired=pd.DataFrame(paired_rows); paired['time_sample']=np.select([paired.quarter.str[:4].astype(int).le(2016),paired.quarter.str[:4].astype(int).le(2020)],['development','validation'],default='final_test')
paired.to_parquet(RESULTS/'paired_quarter_improvements.parquet',index=False)
display(paired.groupby(['representation','timing','time_sample']).normalized_error_reduction.agg(['count','mean','median',lambda x:(x>0).mean(),'min','max']).reset_index())


## 3. HAC and moving-block-bootstrap inference

Rolling five-year targets make neighboring quarters dependent. Newey–West standard errors with four quarterly lags and a four-quarter moving-block bootstrap account for that serial dependence.

In [ ]:
def hac_mean_test(values,lag=4):
    x=np.asarray(values,float); x=x[np.isfinite(x)]; n=len(x); u=x-x.mean(); long_run=float(u@u/n)
    for j in range(1,min(lag,n-1)+1): long_run+=2*(1-j/(lag+1))*float(u[j:]@u[:-j]/n)
    se=np.sqrt(max(long_run,0)/n); return x.mean(),se,x.mean()/se if se>0 else np.nan
def moving_block_interval(values,repetitions=10000,block=4,seed=20260902):
    x=np.asarray(values,float); n=len(x); rng=np.random.default_rng(seed); starts=np.arange(n-block+1); means=[]
    for _ in range(repetitions):
        chosen=rng.choice(starts,size=int(np.ceil(n/block)),replace=True); sample=np.concatenate([x[s:s+block] for s in chosen])[:n]; means.append(sample.mean())
    means=np.asarray(means); return np.quantile(means,[.025,.975]),(1+np.sum(means<=0))/(repetitions+1)
inference=[]
for keys,group in paired.groupby(['representation','timing','time_sample']):
    group=group.sort_values('quarter'); mean,se,t=hac_mean_test(group.normalized_error_reduction); interval,p=moving_block_interval(group.normalized_error_reduction)
    wilcoxon=stats.wilcoxon(group.normalized_error_reduction,alternative='greater')
    inference.append({'representation':keys[0],'timing':keys[1],'time_sample':keys[2],'quarters':len(group),'mean_gain':mean,'hac_se':se,'hac_t':t,'block_bootstrap_lower_95':interval[0],'block_bootstrap_upper_95':interval[1],'block_bootstrap_p_one_sided':p,'wilcoxon_p_one_sided':wilcoxon.pvalue,'positive_quarter_share':group.normalized_error_reduction.gt(0).mean()})
inference=pd.DataFrame(inference); inference.to_parquet(RESULTS/'paired_incremental_inference.parquet',index=False)
display(inference)


## 4. Shuffled-embedding placebo in the final period

For RS-L-Min plus characteristics, all ten embedding coordinates are jointly reassigned among firms within each quarter. Characteristics, targets, splits, preprocessing, penalties, and coverage remain unchanged. One hundred frozen repetitions measure whether arbitrary geometry can reproduce the real increment.

In [ ]:
final_quarters=[q for q in sorted(targets.anchor_quarter.unique()) if int(q[:4])>=2021]
placebo_rows=[]; observed_rows=[]; repetitions=100
for lag,timing in [(1,'economic_q_minus_1'),(2,'public_q_minus_2')]:
    combined_name=f'rs_l_min_10_plus_chars__{timing}'; char_name=f'characteristics_6__{timing}'
    combined_alpha=float(selected.loc[combined_name,'selected_alpha']); char_alpha=float(selected.loc[char_name,'selected_alpha']); quarter_data=[]
    for quarter in final_quarters:
        frame,features,_=erm.load_quarter_frame(ROOT,targets,quarter,'rs_l_min',10,lag,True,False)
        embedding_columns=[c for c in features if c.startswith('embedding_')]; characteristic_columns=[c for c in features if not c.startswith('embedding_')]
        _,actual=erm.fit_predict(frame,features,combined_alpha,fit_splits=('train','validation'),predict_split='test')
        _,chars=erm.fit_predict(frame,characteristic_columns,char_alpha,fit_splits=('train','validation'),predict_split='test')
        observed_rows.append({'timing':timing,'quarter':quarter,'actual_residual_ss':actual['residual_ss'],'characteristic_residual_ss':chars['residual_ss'],'variance_ss':actual['variance_ss']}); quarter_data.append((quarter,frame,features,embedding_columns))
    for repetition in range(repetitions):
        residual=0.0; variance=0.0; rng=np.random.default_rng(20260902+repetition+1000*lag)
        for quarter,frame,features,embedding_columns in quarter_data:
            shuffled=frame.copy(); permutation=rng.permutation(len(shuffled)); shuffled.loc[:,embedding_columns]=shuffled[embedding_columns].to_numpy()[permutation]
            _,summary=erm.fit_predict(shuffled,features,combined_alpha,fit_splits=('train','validation'),predict_split='test'); residual+=summary['residual_ss']; variance+=summary['variance_ss']
        placebo_rows.append({'timing':timing,'repetition':repetition,'placebo_r2':1-residual/variance})
    print(timing,'complete')
placebos=pd.DataFrame(placebo_rows); observed=pd.DataFrame(observed_rows)
placebos.to_parquet(RESULTS/'shuffled_embedding_placebos_100.parquet',index=False); observed.to_parquet(RESULTS/'placebo_observed_components.parquet',index=False)
placebo_summary=[]
for timing,group in placebos.groupby('timing'):
    obs=observed[observed.timing.eq(timing)]; observed_r2=1-obs.actual_residual_ss.sum()/obs.variance_ss.sum(); char_r2=1-obs.characteristic_residual_ss.sum()/obs.variance_ss.sum(); placebo_gain=group.placebo_r2-char_r2; observed_gain=observed_r2-char_r2
    placebo_summary.append({'timing':timing,'observed_combined_r2':observed_r2,'characteristic_r2_same_sample':char_r2,'observed_incremental_gain':observed_gain,'mean_shuffled_incremental_gain':placebo_gain.mean(),'shuffled_95th_percentile_gain':placebo_gain.quantile(.95),'permutation_p_upper':(1+(placebo_gain>=observed_gain).sum())/(1+len(group))})
placebo_summary=pd.DataFrame(placebo_summary); placebo_summary.to_parquet(RESULTS/'shuffled_embedding_placebo_summary.parquet',index=False)
display(placebo_summary)


## 5. Robustness decision gate

The incremental claim advances if RS-L-Min adds positive final-period explanatory power under both timing rules, improves at least 75% of final quarters, has a positive block-bootstrap lower bound under q−1 timing, and beats shuffled geometry.

In [ ]:
def row(timing): return inference.query("representation=='l_min' and timing==@timing and time_sample=='final_test'").iloc[0]
def placebo_row(timing): return placebo_summary[placebo_summary.timing.eq(timing)].iloc[0]
economic=row('economic_q_minus_1'); public=row('public_q_minus_2'); economic_placebo=placebo_row('economic_q_minus_1'); public_placebo=placebo_row('public_q_minus_2')
decision=pd.DataFrame([
 {'test':'q-1 final mean incremental gain positive','value':economic.mean_gain,'passed':economic.mean_gain>0},
 {'test':'q-1 improvement in at least 75% of final quarters','value':economic.positive_quarter_share,'passed':economic.positive_quarter_share>=.75},
 {'test':'q-1 block-bootstrap lower 95% bound positive','value':economic.block_bootstrap_lower_95,'passed':economic.block_bootstrap_lower_95>0},
 {'test':'q-1 shuffled-embedding p <= 0.10','value':economic_placebo.permutation_p_upper,'passed':economic_placebo.permutation_p_upper<=.10},
 {'test':'q-2 final mean incremental gain positive','value':public.mean_gain,'passed':public.mean_gain>0},
 {'test':'q-2 improvement in at least 75% of final quarters','value':public.positive_quarter_share,'passed':public.positive_quarter_share>=.75},
 {'test':'q-2 shuffled-embedding p <= 0.10','value':public_placebo.permutation_p_upper,'passed':public_placebo.permutation_p_upper<=.10}])
decision.to_parquet(RESULTS/'incremental_robustness_gate.parquet',index=False); display(decision)
print('STOP: inspect results before behavioral-embedding and q+1 holdings-adjustment analyses.')
